# 01 — Live seismic intelligence

This notebook retrieves the USGS M2.5+ past-day GeoJSON feed directly from the browser and converts the features into ordinary Python records. The same public feed powers the seismic layer in the standalone dashboard.

> The request is live. Results will differ each time you run it.

In [ ]:
import json
from pyodide.http import open_url

USGS = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_day.geojson'
data = json.load(open_url(USGS))
len(data['features'])

In [ ]:
events = []
for f in data['features']:
    lon, lat, depth = f['geometry']['coordinates']
    p = f['properties']
    events.append({
        'magnitude': p.get('mag'),
        'place': p.get('place'),
        'latitude': lat,
        'longitude': lon,
        'depth_km': depth,
        'url': p.get('url'),
    })

events = sorted(events, key=lambda x: x['magnitude'] or -99, reverse=True)
events[:10]

In [ ]:
from IPython.display import HTML, display

rows = ''.join(
    f"<tr><td>M{e['magnitude']:.1f}</td><td>{e['place']}</td><td>{e['depth_km']:.1f} km</td></tr>"
    for e in events[:12]
)
display(HTML(f'''
<div style="background:#071015;color:#cce2e8;padding:18px;font-family:monospace;border:1px solid #17323c">
<div style="color:#00d4ff;letter-spacing:.16em;margin-bottom:10px">SEISMIC INTEL // USGS LIVE</div>
<table style="width:100%;border-collapse:collapse"><tr style="color:#627a84"><th>MAG</th><th>LOCATION</th><th>DEPTH</th></tr>{rows}</table>
</div>'''))

## Design lesson

WorldView-style dashboards are not only maps: they translate raw feeds into fast-scanning operational cues. Magnitude, timestamp, location, freshness, and source provenance should remain visible enough that the operator can distinguish an event from decoration.